# Artificial intelligence: build a ticket-triage model

**Track:** Foundations · **Guide:** [`005-ai.md`](../guides/concepts/005-ai.md)

AI is a broad field, not a single algorithm. This hands-on lesson builds one
practical AI system: a supervised text classifier that estimates whether a
support ticket needs urgent human review.

We will prepare labeled examples, hold out test data, train a TF-IDF +
logistic-regression pipeline, measure its errors, compare it with a simple
keyword baseline, and score new tickets. The training data is synthetic and
tiny, so this is a learning demo—not a safety or production triage system.

```mermaid
flowchart LR
    A[Labeled example tickets] --> B[Train/test split]
    B --> C[TF-IDF text features]
    C --> D[Train classifier]
    D --> E[Evaluate held-out tickets]
    E --> F[Estimate urgency for a new ticket]
```

**Requirements:** Python and scikit-learn. Install the project
notebook extra as described in [`guides/SETUP.md`](../guides/SETUP.md).
All cells run offline; no API key or cloud account is used.


## 1. Create a small labeled dataset

Each ticket has an input text and a human-assigned label. A real system
would need carefully reviewed, representative, privacy-approved examples.
We use synthetic examples here so the exercise is reproducible and private.


In [ ]:
from collections import Counter

tickets = [
    ("Someone threatened to hurt me and knows where I live", "urgent"),
    ("I may hurt myself and need help immediately", "urgent"),
    ("An attacker broke into my account and is stealing funds", "urgent"),
    ("Smoke is coming from the device and it is getting hot", "urgent"),
    ("I was injured when the charger exploded", "urgent"),
    ("Someone is following me after sending threats", "urgent"),
    ("An intruder has my password and is taking money", "urgent"),
    ("My device sparked and burned my hand", "urgent"),
    ("The battery is swelling and smells like burning", "urgent"),
    ("I received a threat that someone will attack me", "urgent"),
    ("My account was hijacked and unauthorized transfers started", "urgent"),
    ("I feel unsafe because my partner threatened violence", "urgent"),
    ("I am thinking about harming myself tonight", "urgent"),
    ("The adapter caught fire and damaged the desk", "urgent"),
    ("Someone stole my login and changed all recovery details", "urgent"),
    ("My child got an electric shock from the product", "urgent"),
    ("How can I update my billing address", "routine"),
    ("Please explain the monthly subscription charge", "routine"),
    ("Where can I download the user manual", "routine"),
    ("Can I change my order before it ships", "routine"),
    ("The screen arrived scratched but still turns on", "routine"),
    ("I forgot my password and need to reset it", "routine"),
    ("Please cancel the renewal for next month", "routine"),
    ("How long does standard delivery take", "routine"),
    ("The app is slow when I open settings", "routine"),
    ("I would like to return the wrong color item", "routine"),
    ("Can I transfer my saved settings to a new phone", "routine"),
    ("I received a duplicate receipt by email", "routine"),
    ("The package is late and tracking has not changed", "routine"),
    ("How do I pair the headphones to my laptop", "routine"),
    ("The invoice has an incorrect company name", "routine"),
    ("The keyboard key is sticky but the rest works", "routine"),
]

texts = [text for text, _label in tickets]
labels = [label for _text, label in tickets]
print(f"Examples: {len(tickets)}")
print(f"Label counts: {dict(Counter(labels))}")
print(f"One example: {tickets[0]}")


## 2. Split before learning

Keep some labeled examples out of training. The test set estimates
performance on examples the model did not fit on. `stratify` preserves the
class balance in this small split; `random_state` makes it repeatable.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels,
)
print(f"Training examples: {len(X_train)}")
print(f"Held-out test examples: {len(X_test)}")
print(f"Test label counts: {dict(Counter(y_test))}")


## 3. Build and train a machine-learning model

TF-IDF turns ticket text into weighted word and phrase features. Logistic
regression learns weights from the labeled training examples. Keeping
vectorization and classification in one scikit-learn `Pipeline` ensures
the vocabulary is fit on training data only and reused consistently for
evaluation and predictions.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True),
    LogisticRegression(max_iter=1000, random_state=42),
)
model.fit(X_train, y_train)
print("Learned labels:", list(model.classes_))
print("Vocabulary size:", len(model.named_steps["tfidfvectorizer"].vocabulary_))


## 4. Evaluate on tickets the model did not train on

Accuracy is only the fraction correct. The classification report also
exposes precision and recall for each label. For a real safety-sensitive
workflow, a missed urgent case can be much more costly than an extra
human review, so inspect urgent-class recall and every error—not accuracy
alone. This tiny holdout is illustrative and cannot establish readiness.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

test_predictions = model.predict(X_test)
print(f"Held-out accuracy: {accuracy_score(y_test, test_predictions):.2f}")
print(classification_report(y_test, test_predictions, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_test, test_predictions, labels=model.classes_))
print("Label order:", list(model.classes_))


## 5. Compare against a hand-written baseline

A keyword rule is easier to explain, but only recognizes words explicitly
listed by its author. Compare its held-out predictions with the learned
model. Neither approach is automatically correct: look at missed urgent
tickets and false alarms before choosing an approach.


In [ ]:
from sklearn.metrics import precision_score, recall_score

urgent_keywords = {
    "hurt", "harm", "threat", "smoke", "fire", "injured",
    "burned", "burning", "unsafe", "hijacked", "stole",
    "stealing", "shock", "attacker", "exploded",
}

def keyword_baseline(text):
    words = set(text.lower().replace(",", " ").replace(".", " ").split())
    return "urgent" if words & urgent_keywords else "routine"

rule_predictions = [keyword_baseline(text) for text in X_test]
for name, predictions in (
    ("keyword rule", rule_predictions),
    ("TF-IDF + logistic regression", test_predictions),
):
    print(
        f"{name}: "
        f"accuracy={accuracy_score(y_test, predictions):.2f}, "
        f"urgent precision={precision_score(y_test, predictions, pos_label='urgent', zero_division=0):.2f}, "
        f"urgent recall={recall_score(y_test, predictions, pos_label='urgent', zero_division=0):.2f}"
    )


## 6. Score new tickets

`predict_proba` gives model scores, not guaranteed probabilities of real-world
risk. The example uses a configurable 0.50 review threshold to demonstrate a
human-in-the-loop policy. Never use this tiny synthetic model to make real
safety, account, or emergency decisions.


In [ ]:
from pprint import pprint


def triage_ticket(text, review_threshold=0.50):
    probabilities = model.predict_proba([text])[0]
    urgent_index = list(model.classes_).index("urgent")
    urgent_score = float(probabilities[urgent_index])
    return {
        "ticket": text,
        "urgent_model_score": round(urgent_score, 3),
        "predicted_label": model.classes_[probabilities.argmax()],
        "action": (
            "human_review"
            if urgent_score >= review_threshold
            else "standard_queue"
        ),
    }

new_tickets = [
    "There is a burning smell and smoke coming from my charger",
    "How can I download a copy of my monthly invoice?",
]
predictions = [triage_ticket(text) for text in new_tickets]
pprint(predictions, sort_dicts=False)


## 7. Inspect what the text model learned

A linear model associates learned word features with each class. The
weights are a useful debugging aid, not a causal explanation or guarantee
that a prediction is fair. Check for spurious shortcuts and data leakage.


In [ ]:
tfidf = model.named_steps["tfidfvectorizer"]
classifier = model.named_steps["logisticregression"]
feature_names = tfidf.get_feature_names_out()
urgent_weights = classifier.coef_[0]

most_urgent = sorted(
    zip(urgent_weights, feature_names), reverse=True
)[:10]
most_routine = sorted(
    zip(urgent_weights, feature_names)
)[:10]
print("Features associated with urgent:")
print([feature for _weight, feature in most_urgent])
print("Features associated with routine:")
print([feature for _weight, feature in most_routine])


## What this example does—and does not—show

This is a complete **small supervised-learning workflow**, an important
branch of AI: labeled data → train/test split → feature extraction → model
fitting → held-out evaluation → inference. The classifier learns patterns
from examples; it is more than a hand-authored `if` statement.

It is not a general reasoning system. The tiny synthetic dataset cannot
represent real language, class imbalance, changing behavior, or the costs
of mistakes. Do not deploy it for actual safety triage. A responsible
system needs reviewed representative data, privacy controls, stronger
evaluation, monitored human review, and a tested escalation process.

**Exercises**

1. Add several labeled examples of your own, retrain, and note the test
   metrics. With this small sample, metrics will vary as the dataset changes.
2. Inspect every false negative. Which one would have the highest impact?
3. Change the human-review threshold and explain the precision/recall trade-off.
4. Compare a rule, this learned classifier, and a language-model approach:
   what data, latency, explainability, and failure controls would each need?

Continue with the [AI concept guide](../guides/concepts/005-ai.md) and the
[learning roadmap](../guides/ROADMAP.md).
